# GISLR — windows, two offset models, ensembles, staggered restarts + next-gloss prior (TODO §12.8)

**Diagnostic, not a pipeline stage. Inference only — no training.**

The user's proposal (2026-09-26): instead of a model reading the stream frame by frame, feed windows of about
a sign's length (1-3 s) to **two models offset by ~500 ms**, combine their predictions, and fuse them with an
independent **next-gloss model** trained on gloss order. This notebook scores that idea on GISLR-Sentences next
to C1, together with the two variants Claude suggested (per-frame ensembles of two continuous models, and
two copies of one model restarted in turn):

| arm | what reads the stream | decoder |
|---|---|---|
| `win{1,2,3}s_one` | one isolated `gru` (ME_132 xy), consecutive windows of 1/2/3 s | runs of the same confident top gloss (τ, min run) |
| `win…_offset_same` | the same `gru` twice, the second 500 ms later: **the user's scheme with one model** | same, on the two series merged (each point = mean of both models' latest window) |
| `win…_offset_pair` | `gru` + `gru_phono_raw` (the best isolated pair), 500 ms apart: **the user's scheme, two models** | same |
| `win…_dense_pair` | the same pair, each a new window every 500 ms (offset 250 ms): the dense version | same |
| `C1` | C1 per frame (the deployed model; reference) | D3 (ν, min_len) |
| `C1+C2` (…`C4+C5` once trained) | two continuous models, per-frame probabilities averaged | D3 |
| `C1_stagger{300,600}_{mean,older}` | two copies of C1, each restarted every 10/20 s, half a period apart | D3 |

Every arm is decoded **without** and **with the next-gloss prior** (the app's: a held-out trigram over the
sentence corpus, sentence-fold, fused as `q·p^λ`, `sb.recognize.continuous.fuse`). Protocol as every §12
evaluation: settings chosen on the **5 selection signers** (first the segmenter, then the fusion rule),
reported on the **16 evaluation signers**, `sentence` streams, repeats collapsed. Also reported with the same
settings: **hard-cut** (no pause between signs) and **long sessions** (6 sentences chained, no prior: drift).

| artifact | path |
|---|---|
| per-arm results (resumable, one file per arm) | `data/cache/gislr/window_ensemble/parts/<arm>.json` |
| summary table | `data/cache/gislr/window_ensemble/summary.csv` |
| figure | `data/cache/gislr/window_ensemble/assets/ger_by_arm.png` |

**Resumable:** an arm whose part file exists is skipped. Continuous runs that are not trained yet (C4, C5) are
skipped; re-run after training them and only their arms are computed.

## 1. Setup

In [1]:
# ============================================================
# Imports, tunables, streams, signer split, models, next-gloss prior
# ============================================================
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, REGISTRY_DIR, gislr_sentences_dir
from sb.recognize import streaming as S
from sb.recognize.continuous import fuse as FU
from sb.recognize.continuous.decode import frame_outputs
from sb.recognize.continuous.train import load_run, run_dir_for
from sb.recognize.sequences import baselines as B
from sb.recognize.sequences import metrics as M
from sb.recognize.sequences import windows as W
from sb.recognize.sequences.corpus import load_sentences
from sb.rescore import prior as P

DOWN = json.loads((EXPERIMENTS_DIR / "recognition" / "configs" / "gislr.downstream.json").read_text(encoding="utf-8"))

LIMIT_PER_GROUP = None  # e.g. 10 -> smoke run, written to window_ensemble_smoke/
SEED, N_SELECTION_SIGNERS = 42, 5  # the §12 signer split
OFFSET = 15               # frames (500 ms at 30 fps): the user's offset between the two window models
WINDOWS = [30, 60, 90]    # 1, 2, 3 s
DENSE_STRIDE = 15         # dense arms: each model starts a window every 500 ms
ISOLATED = {"gru": "1789559734", "gru_phono_raw": "1790355555"}  # best raw / best phono+raw streaming models
CONTINUOUS = ["C1", "C2", "C4", "C5"]   # untrained runs are skipped
PAIRS = [("C1", "C2"), ("C4", "C5"), ("C1", "C4")]
STAGGER = {"C1": [300, 600], "C4": [300, 600]}  # restart periods (10 s, 20 s)
TAUS, MIN_RUNS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8], [1, 2]
NUS, MIN_LENS = [0.5, 0.6, 0.7, 0.8], [2, 4, 6]
PRIOR_RULES = [FU.Rule("rescore", lam=lam, theta=th) for lam in (0.3, 0.5, 1.0) for th in (0.0, 0.2)]
LM_ORDER = 3              # trigram, the app's prior
CHAIN = 6                 # sentences per long session

OUT = CACHE_DIR / "gislr" / ("window_ensemble" if LIMIT_PER_GROUP is None else "window_ensemble_smoke")
PARTS, ASSETS = OUT / "parts", OUT / "assets"
for d in (PARTS, ASSETS):
    d.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

ROOT = gislr_sentences_dir("v1")
SEQ = pd.read_csv(ROOT / "sequences.csv", keep_default_na=False)
SEQ["row"] = np.arange(len(SEQ))
SELECT, EVAL = B.signer_split(SEQ["participant_id"], N_SELECTION_SIGNERS, SEED)
SENT = SEQ[SEQ.kind == "sentence"].copy()
SENT["group"] = np.where(SENT.participant_id.isin(SELECT), "select", "eval")
if LIMIT_PER_GROUP is not None:
    SENT = SENT.groupby("group", group_keys=False).head(LIMIT_PER_GROUP)
ROWS = {g: SENT.loc[SENT.group == g, "row"].tolist() for g in ("select", "eval")}
CHAINS = [r[j:j + CHAIN] for _, g in SENT[SENT.group == "eval"].groupby("participant_id")
          for r in [g["row"].tolist()] for j in range(0, len(r) - CHAIN + 1, CHAIN)]

MODELS = {}
for name, rid in ISOLATED.items():
    lm = B.load_registry_model(REGISTRY_DIR / "runs" / rid, 250, DEVICE)
    MODELS[name] = {"model": lm.model, "kind": "isolated", "run": rid,
                    "spec": (tuple(int(r) for r in lm.rows), lm.coords)}
for name in CONTINUOUS:
    rd = run_dir_for(name)
    if rd is None or not (rd / "meta.json").exists() or not json.loads((rd / "meta.json").read_text())["training"]["finished"]:
        print(f"{name}: not trained -- its arms are skipped")
        continue
    m, ck = load_run(rd, DEVICE)
    assert ck["null_index"] == 250
    MODELS[name] = {"model": m, "kind": "continuous", "run": rd.name,
                    "spec": (tuple(int(r) for r in ck["landmarks"]), ck["coords"])}
FEATS = {}
for v in MODELS.values():
    if v["spec"] not in FEATS:
        FEATS[v["spec"]] = B.StreamFeatures.build(ROOT, SEQ, np.asarray(v["spec"][0]), v["spec"][1])
NULL = 250

# next-gloss prior: trigram per sentence-fold, never fitted on the sentence it scores (as gislr.4.downstream)
LABEL_MAP = json.loads((ROOT / "sign_to_prediction_index_map.json").read_text(encoding="utf-8"))
IDX2GLOSS = [g for g, _ in sorted(LABEL_MAP.items(), key=lambda kv: kv[1])]
SENTS = load_sentences(DOWN["corpus_version"])
GLOSSES = sorted({g for s in SENTS for g in s.glosses})
PERM = np.array([GLOSSES.index(g) for g in IDX2GLOSS])
FOLD = P.sentence_folds([s.id for s in SENTS], [s.theme for s in SENTS], DOWN["lm"]["n_folds"], DOWN["seed"])
_PRIORS: dict = {}


def prior_for(row: int):
    fold = FOLD[SEQ.at[row, "sentence_id"]]
    if fold not in _PRIORS:
        lm = P.NgramLM(GLOSSES, LM_ORDER, DOWN["lm"]["discount"]).fit([s.glosses for s in SENTS if FOLD[s.id] != fold])
        memo: dict = {}

        def prior(hist, lm=lm, memo=memo):
            if hist not in memo:
                memo[hist] = lm.gloss_dist([IDX2GLOSS[c] for c in hist])[PERM]
            return memo[hist]
        _PRIORS[fold] = prior
    return _PRIORS[fold]


print("models:", {k: f"{v['kind']} {v['run']} {len(v['spec'][0])}{v['spec'][1]}" for k, v in MODELS.items()})
print(f"streams: select {len(ROWS['select'])} · eval {len(ROWS['eval'])} · long sessions {len(CHAINS)} x {CHAIN}")
print(f"results -> {OUT}")

gislr_sentences_dir: kaggle copy unavailable (KaggleApiHTTPError); using the local build C:\Users\user2\repository\data\cache\gislr\sentences\v1


C4: not trained -- its arms are skipped
C5: not trained -- its arms are skipped


models: {'gru': 'isolated 1789559734 132xy', 'gru_phono_raw': 'isolated 1790355555 134xyz', 'C1': 'continuous 1790143122 132xy', 'C2': 'continuous 1790144582 132xy'}
streams: select 1562 · eval 5054 · long sessions 835 x 6
results -> C:\Users\user2\repository\data\cache\gislr\window_ensemble


## 2. Arms

An arm turns one stream into segments. Window arms keep, per stream, the merged series of window
predictions; frame arms keep per-frame gloss+null probabilities. Both are computed once per stream and
decoded for every setting.

In [2]:
# ============================================================
# Arm definitions: raw outputs per stream -> segments per setting
# ============================================================
ARMS = {}
for w in WINDOWS:
    s = f"win{w // 30}s"
    ARMS[f"{s}_one"] = {"type": "window", "win": w, "members": [("gru", 0, w)]}
    ARMS[f"{s}_offset_same"] = {"type": "window", "win": w, "members": [("gru", 0, w), ("gru", OFFSET, w)]}
    ARMS[f"{s}_offset_pair"] = {"type": "window", "win": w, "members": [("gru", 0, w), ("gru_phono_raw", OFFSET, w)]}
    ARMS[f"{s}_dense_pair"] = {"type": "window", "win": w,
                               "members": [("gru", 0, DENSE_STRIDE), ("gru_phono_raw", DENSE_STRIDE // 2, DENSE_STRIDE)]}
for name in CONTINUOUS:
    ARMS[name] = {"type": "frame", "members": [name]}
for a, b in PAIRS:
    ARMS[f"{a}+{b}"] = {"type": "frame", "members": [a, b]}
for name, periods in STAGGER.items():
    for per in periods:
        for comb in ("mean", "older"):
            ARMS[f"{name}_stagger{per}_{comb}"] = {"type": "frame", "members": [name], "stagger": (per, comb)}
ARMS = {k: v for k, v in ARMS.items() if all((m[0] if isinstance(m, tuple) else m) in MODELS for m in v["members"])}


def stream(rows: list[int], variant: str) -> dict:
    """One stream (or a chain of streams) for every input spec: x per spec, frame kinds, labels, segments."""
    xs = {spec: [] for spec in FEATS}
    kinds, labels, segs, off = [], [], [], 0
    for row in rows:
        lab, seg = B.seq_arrays(SEQ.iloc[row])
        k = FEATS[next(iter(FEATS))].kind(row)
        for spec, f in FEATS.items():
            x = f.x(row)
            if variant == "hardcut":
                x, _, _ = B.hard_cut(x, k, seg)
            xs[spec].append(x)
        if variant == "hardcut":
            _, k, seg = B.hard_cut(np.zeros((len(k), 1)), k, seg)
        kinds.append(k)
        labels.append(lab)
        segs.append(seg + off)
        off += len(k)
    return {"x": {spec: torch.from_numpy(np.ascontiguousarray(np.concatenate(v))).to(DEVICE) for spec, v in xs.items()},
            "kinds": np.concatenate(kinds), "labels": np.concatenate(labels), "seg": np.concatenate(segs),
            "row": rows[0]}


def raw(arm: dict, st: dict):
    if arm["type"] == "window":
        series = [W.window_probs(MODELS[m]["model"], st["x"][MODELS[m]["spec"]], arm["win"], stride, off)
                  for m, off, stride in arm["members"]]
        e, p = series[0] if len(series) == 1 else W.combine_offset(*series[0], *series[1])
        return e, p.astype(np.float16)
    outs = []
    for m in arm["members"]:
        x = st["x"][MODELS[m]["spec"]]
        if "stagger" in arm:
            outs.append(W.staggered(MODELS[m]["model"], x, *arm["stagger"]))
        else:
            outs.append(frame_outputs(MODELS[m]["model"], x)[0])
    return W.average(*outs).astype(np.float16)


def grid(arm: dict) -> list[tuple]:
    return ([(t, r) for t in TAUS for r in MIN_RUNS] if arm["type"] == "window"
            else [(n, m) for n in NUS for m in MIN_LENS])


def segments(arm: dict, out, params: tuple) -> list:
    if arm["type"] == "window":
        e, p = out
        return W.window_segments(e, p.astype(np.float32), params[0], arm["win"], params[1])
    return FU.segments_d3(out.astype(np.float32), params[0], params[1], NULL)


def score(items: list[tuple], rule: FU.Rule, use_prior: bool) -> dict:
    """items: (segments, stream) pairs -> aggregate over them."""
    sc = []
    for segs, st in items:
        em = S.collapse_repeats(FU.decode_fused(segs, prior_for(st["row"]) if use_prior else None, rule))
        sc.append(M.score_sequence(em, st["labels"], st["seg"], st["kinds"]))
    return M.aggregate(sc, B.frame_totals([st["kinds"] for _, st in items]))


print(f"{len(ARMS)} arms:", ", ".join(ARMS))

19 arms: win1s_one, win1s_offset_same, win1s_offset_pair, win1s_dense_pair, win2s_one, win2s_offset_same, win2s_offset_pair, win2s_dense_pair, win3s_one, win3s_offset_same, win3s_offset_pair, win3s_dense_pair, C1, C2, C1+C2, C1_stagger300_mean, C1_stagger300_older, C1_stagger600_mean, C1_stagger600_older


## 3. Select on the 5 selection signers, score the 16 evaluation signers

Per arm: the segmenter setting with the lowest selection GER without a prior, then the fusion rule with the
lowest selection GER on that setting. The evaluation signers are then scored with both (no prior / prior) on
clean and hard-cut streams, and without a prior on the long sessions. The C1 arm also reports its GER at the
continuous evaluation's own setting (D3 ν 0.7, min_len 4) as a parity check against 0.293.

In [3]:
# ============================================================
# Per arm: select, then evaluate (one part file per arm, resumable)
# ============================================================
KEEP = ("ger", "sub_rate", "del_rate", "ins_rate", "sentence_acc", "latency_median", "n_ref")


def slim(agg: dict) -> dict:
    return {k: agg[k] for k in KEEP}


for name, arm in ARMS.items():
    path = PARTS / f"{name}.json"
    if path.exists():
        continue
    sel = [(raw(arm, st), st) for st in (stream([r], "clean") for r in tqdm(ROWS["select"], desc=f"{name} select", leave=False))]
    for _, st in sel:
        st["x"] = None  # keep the outputs, drop the inputs
    g = {p: score([(segments(arm, o, p), st) for o, st in sel], FU.Rule("none"), False)["ger"] for p in grid(arm)}
    best = min(g, key=g.get)
    segs_sel = [(segments(arm, o, best), st) for o, st in sel]
    gr = {i: score(segs_sel, r, True)["ger"] for i, r in enumerate(PRIOR_RULES)}
    rule = PRIOR_RULES[min(gr, key=gr.get)]
    del sel, segs_sel
    res = {"arm": name, "type": arm["type"], "params": list(best), "rule": rule.as_dict(),
           "select_ger_none": g[best], "select_ger_prior": min(gr.values()), "eval": {}}
    for variant, groups in (("clean", [[r] for r in ROWS["eval"]]), ("hardcut", [[r] for r in ROWS["eval"]]), ("long", CHAINS)):
        items, parity = [], []
        for rows in tqdm(groups, desc=f"{name} eval {variant}", leave=False):
            st = stream(rows, "hardcut" if variant == "hardcut" else "clean")
            o = raw(arm, st)
            st["x"] = None
            items.append((segments(arm, o, best), st))
            if name == "C1" and variant == "clean":
                parity.append((segments(arm, o, (0.7, 4)), st))
        res["eval"][f"{variant}/none"] = slim(score(items, FU.Rule("none"), False))
        if variant != "long":  # a chain spans sentences of several corpus folds: no held-out prior for it
            res["eval"][f"{variant}/prior"] = slim(score(items, rule, True))
        if parity:
            res["parity_ger_c1_d3_0.7_4"] = score(parity, FU.Rule("none"), False)["ger"]
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(res, indent=1, default=float))
    os.replace(tmp, path)
    print(f"{name}: params {best} · eval clean GER {res['eval']['clean/none']['ger']:.3f} "
          f"-> {res['eval']['clean/prior']['ger']:.3f} with prior")

win1s_one select:   0%|          | 0/1562 [00:00<?, ?it/s]

win1s_one eval clean:   0%|          | 0/5054 [00:00<?, ?it/s]

win1s_one eval hardcut:   0%|          | 0/5054 [00:00<?, ?it/s]

win1s_one eval long:   0%|          | 0/835 [00:00<?, ?it/s]

win1s_one: params (0.4, 1) · eval clean GER 0.609 -> 0.605 with prior


...

C1_stagger600_older select:   0%|          | 0/1562 [00:00<?, ?it/s]

C1_stagger600_older eval clean:   0%|          | 0/5054 [00:00<?, ?it/s]

C1_stagger600_older eval hardcut:   0%|          | 0/5054 [00:00<?, ?it/s]

C1_stagger600_older eval long:   0%|          | 0/835 [00:00<?, ?it/s]

C1_stagger600_older: params (0.7, 4) · eval clean GER 0.293 -> 0.275 with prior


## 4. Results

In [4]:
# ============================================================
# Summary table + figure
# ============================================================
rows_ = []
for p in sorted(PARTS.glob("*.json")):
    r = json.loads(p.read_text())
    row = {"arm": r["arm"], "params": r["params"], "lam": r["rule"]["lam"], "theta": r["rule"]["theta"],
           "select_none": r["select_ger_none"], "select_prior": r["select_ger_prior"]}
    for k, v in r["eval"].items():
        row[f"{k} GER"] = v["ger"]
        if k in ("clean/none", "clean/prior"):
            for m in ("sub_rate", "del_rate", "ins_rate", "sentence_acc", "latency_median"):
                row[f"{k} {m}"] = v[m]
    if "parity_ger_c1_d3_0.7_4" in r:
        print(f"parity: C1 D3 (0.7, 4) on the evaluation signers = {r['parity_ger_c1_d3_0.7_4']:.4f} (continuous eval: 0.2934)")
    rows_.append(row)
SUM = pd.DataFrame(rows_).set_index("arm").sort_values("clean/prior GER")
SUM.to_csv(OUT / "summary.csv")
cols = ["params", "clean/none GER", "clean/prior GER", "hardcut/none GER", "hardcut/prior GER", "long/none GER",
        "clean/prior del_rate", "clean/prior ins_rate", "clean/prior sentence_acc", "clean/prior latency_median"]
print(SUM[[c for c in cols if c in SUM]].to_string(float_format="{:.3f}".format))

fig, ax = plt.subplots(figsize=(9, 0.3 * len(SUM) + 1.5))
yy = np.arange(len(SUM))
for k, (col, lab) in enumerate((("clean/none GER", "clean, no prior"), ("clean/prior GER", "clean + next-gloss prior"),
                                 ("hardcut/prior GER", "no pauses + prior"))):
    ax.barh(yy - 0.27 + 0.27 * k, SUM[col], height=0.27, label=lab)
ax.set_yticks(yy, SUM.index)
ax.invert_yaxis()
ax.set_xlabel("gloss error rate, 16 evaluation signers (lower is better)")
ax.legend(loc="lower right", fontsize=8)
ax.set_title("Windows + offset models vs per-frame models, with and without the next-gloss prior")
fig.tight_layout()
fig.savefig(ASSETS / "ger_by_arm.png", dpi=130)
plt.show()

parity: C1 D3 (0.7, 4) on the evaluation signers = 0.2934 (continuous eval: 0.2934)
                       params  clean/none GER  clean/prior GER  hardcut/none GER  hardcut/prior GER  long/none GER  clean/prior del_rate  clean/prior ins_rate  clean/prior sentence_acc  clean/prior latency_median
arm                                                                                                                                                                                                                 
C1+C2                [0.5, 4]           0.262            0.244             0.452              0.446          0.451                 0.040                 0.020                     0.465                       1.000
C1_stagger600_older  [0.7, 4]           0.293            0.275             0.546              0.542          0.416                 0.062                 0.017                     0.407                       1.000
C1                   [0.7, 4]           0.293            0.276  

C:\Users\user2\AppData\Local\Temp\ipykernel_16960\4132073749.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
